# Vehicles

In [0]:
from pyspark.sql.functions import *

vehicles_clean = spark.table("vehicle_insurance.bronze.vehicles") \
    .dropDuplicates(["vehicle_id"]) \
    .withColumn("vehicle_type", initcap(trim(col("vehicle_type")))) \
    .withColumn("brand", initcap(trim(col("brand")))) \
    .withColumn("model", initcap(trim(col("model")))) \
    .withColumn("year", col("year").cast("int")) \
    .withColumn("last_updated", to_timestamp(col("last_updated"), "dd-MM-yyyy HH:mm")) \
    .filter(col("vehicle_id").isNotNull()) \
    .filter(col("year") <= year(current_date()))   # remove future invalid years

In [0]:
vehicles_clean=vehicles_clean.dropna()

In [0]:
from pyspark.sql.functions import col, sum

vehicles_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in vehicles_clean.columns
]).show()

In [0]:
vehicles_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_insurance.silver.vehicles")

#Customers

In [0]:
from pyspark.sql.functions import *

customers_clean = spark.table("vehicle_insurance.bronze.customers") \
    .dropDuplicates(["customer_id"]) \
    .withColumn("name", initcap(trim(col("name")))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("state", initcap(trim(col("state")))) \
    .withColumn("risk_category", upper(trim(col("risk_category")))) \
    .withColumn("last_updated", to_timestamp(col("last_updated"), "dd-MM-yyyy HH:mm")) \
    .fillna({"city": "Unknown"}) \
    .filter(col("customer_id").isNotNull())

In [0]:
from pyspark.sql.functions import col, sum

customers_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customers_clean.columns
]).show()

In [0]:
customers_clean=customers_clean.dropna()

In [0]:
customers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_insurance.silver.customers")

#payments

In [0]:
from pyspark.sql.functions import *

payments_clean = spark.table("vehicle_insurance.bronze.payments") \
    .dropDuplicates(["payment_id"]) \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("status", upper(trim(col("status")))) \
    .filter(col("payment_id").isNotNull()) \
    .filter(col("claim_id").isNotNull()) \
    .filter(col("amount") > 0)

In [0]:
from pyspark.sql.functions import col, sum

payments_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in payments_clean.columns
]).show()

In [0]:
payments_clean=payments_clean.dropna()

In [0]:
payments_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_insurance.silver.payments")

#Policies

In [0]:
from pyspark.sql.functions import *

policies_clean = spark.table("vehicle_insurance.bronze.policies") \
    .dropDuplicates(["policy_id"]) \
    .withColumn("premium_amount", col("premium_amount").cast("double")) \
    .withColumn("status", upper(trim(col("status")))) \
    .filter(col("policy_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .filter(col("premium_amount") > 0)

In [0]:
from pyspark.sql.functions import col, sum

policies_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in policies_clean.columns
]).show()

In [0]:
policies_clean=policies_clean.dropna()

In [0]:
policies_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_insurance.silver.policies")

#fnol


In [0]:
from pyspark.sql.functions import *

fnol_clean = spark.table("vehicle_insurance.bronze.fnol") \
    .dropDuplicates(["event_id"]) \
    .withColumn("event_id", trim(col("event_id"))) \
    .withColumn("claim_id", trim(col("claim_id"))) \
    .withColumn("event_type", upper(trim(col("event_type")))) \
    .withColumn("description", initcap(trim(col("description")))) \
    .withColumn("event_time", to_timestamp(col("event_time"), "dd-MM-yyyy HH:mm")) \
    .filter(col("event_id").isNotNull()) \
    .filter(col("claim_id").isNotNull()) \
    .filter(col("event_time").isNotNull())

In [0]:
from pyspark.sql.functions import col, sum

fnol_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in fnol_clean.columns
]).show()

In [0]:
fnol_clean=fnol_clean.dropna()

In [0]:
fnol_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_insurance.silver.fnol")

#claims

In [0]:
from pyspark.sql.functions import *

claims_clean = spark.table("vehicle_insurance.bronze.claims") \
    .dropDuplicates(["claim_id"]) \
    .withColumn("claim_id", trim(col("claim_id"))) \
    .withColumn("policy_id", trim(col("policy_id"))) \
    .withColumn("claim_amount", col("claim_amount").cast("double")) \
    .withColumn("claim_status", upper(trim(col("claim_status")))) \
    .withColumn("claim_date", to_timestamp(col("claim_date"), "dd-MM-yyyy HH:mm")) \
    .filter(col("claim_id").isNotNull()) \
    .filter(col("policy_id").isNotNull()) \
    .filter(col("claim_amount").isNotNull()) \
    .filter(col("claim_date").isNotNull()) \
    .filter(col("claim_amount") >= 0)   # remove invalid negative amounts

In [0]:
from pyspark.sql.functions import col, sum

claims_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in claims_clean.columns
]).show()

In [0]:
claims_clean=claims_clean.dropna()

In [0]:
claims_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vehicle_insurance.silver.claims")